# C4 Notebook 03 — ASCAT SWI V4 (daily): pilot → bulk → QA

**Purpose.** Produce independently versioned, *target-free* SWI001/SWI005/SWI010 soil-wetness features for all **1,429 station-day records**. This is the satellite auxiliary branch, *not* an image and not the final 128-D neural representation.

**Official product:** [Copernicus Land Monitoring Service daily global SWI V4](https://land.copernicus.eu/en/products/soil-moisture/daily-soil-water-index-global-12-5km-v4), 0.1° WGS84 grid; [Sentinel Hub product access specification](https://documentation.dataspace.copernicus.eu/APIs/SentinelHub/Data/clms/bio-geophysical-parameters/soil-moisture/soil-water-index/swi_global_12.5km_daily_v4.html). It is **not** a Google Earth Engine asset. Access in this notebook is through **Copernicus Data Space Sentinel Hub OAuth + Process API** using public BYOC collection `59c08cd0-762b-4b8a-9eaa-0d55d054a732`.

**Source bands:** `SWI001/SWI005/SWI010`, matching `QFLAG001/QFLAG005/QFLAG010`, `SSF`, and `dataMask`. Stored SWI and QFLAG digital numbers are decoded by ×0.5 to obtain percent, while fill value 255 is rejected *before* scaling. SSF=1 means unfrozen. See the [SWI V4 product manual](https://land.copernicus.eu/en/technical-library/product-user-manual-soil-water-index-v4-0.1deg/@@download/file), especially Sections 3.1.1 and 5.1.2.

**Temporal contract:** For forecast origin **Day T, 09:00 Asia/Colombo**, use only product dates **T−1 or earlier**. The saved `swi_age_days` is **calendar product-date lag**, *not* the time since the last physical ASCAT overpass nor proof of publication by 09:00. Retrospective hindcast only.

**Spatial contract:** Native-*scale* 0.1° WGS84 grid (source global grid edges align with whole tenths). Download each national-scale historical day **once**. Intersect original irregular upstream polygons with native-sized cells; use equal-area overlap weights on valid cells. If insufficient catchment coverage, attempt a **flagged nearest valid grid-cell fallback** near the original gauge. No upsampling or fabricated local wetness; no target labels loaded.

In [ ]:
# Cell 1 — Imports, private paths, source contract and safety switches
from pathlib import Path
from datetime import date, datetime, timedelta
from zoneinfo import ZoneInfo
import hashlib, io, json, math, os, tempfile, time
import requests
import numpy as np
import pandas as pd
from shapely.geometry import shape, box
from shapely.ops import transform as geometry_transform
from pyproj import Transformer, Geod
from rasterio.io import MemoryFile
from rasterio.transform import from_origin
from IPython.display import display

REPO_ROOT_OVERRIDE = None
PILOT_STATION = 'holombuwa'
PILOT_FORECAST_DATE = '2025-11-17'
RUN_PILOT = True          
RUN_BULK = True
MAX_NEW_DAYS_PER_RUN = 3   # Restartable batches: one API request per product date

# All values below are provisional PP1 data policy; do not change during a run.
MAX_SWI_AGE_DAYS = 7              # Search products T-1 ... T-7 inclusive
MIN_QFLAG_PERCENT = 20.0          # Provisional per-band quality threshold, NOT publisher universal cutoff
MIN_CATCHMENT_VALID_AREA = 0.50   # >=50% of true polygon area must have jointly valid pixels
MAX_NEAREST_DISTANCE_KM = 25.0    # Explicitly flagged nearest-native-cell fallback
SWI_COLLECTION_ID = '59c08cd0-762b-4b8a-9eaa-0d55d054a732'
PROCESS_URL = 'https://sh.dataspace.copernicus.eu/process/v1'
TOKEN_URL = 'https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token'
FEATURES = ['swi001_percent', 'swi005_percent', 'swi010_percent']
QUALITY = ['qflag001_percent','qflag005_percent','qflag010_percent']
BANDS = FEATURES + QUALITY + ['ssf', 'data_mask']
SOURCE_VERSION = 'C4_ASCAT_SWIV4_V1_DAY_MINUS1_NATIVE_01DEG_JOINTQA'
SAMPLE_TZ = ZoneInfo('Asia/Colombo')


def find_root():
    if REPO_ROOT_OVERRIDE:
        root = Path(REPO_ROOT_OVERRIDE).expanduser().resolve()
        if (root/'c4_fusion').is_dir() and (root/'shared/spatial/catchments').is_dir(): return root
        raise FileNotFoundError('Invalid repository root override')
    for p0 in [Path.cwd().resolve(), Path.cwd().resolve().parent]:
        for p in [p0,*p0.parents]:
            if (p/'c4_fusion').is_dir() and (p/'shared/spatial/catchments').is_dir(): return p
    raise FileNotFoundError('Cannot find J26-DS-314. Set REPO_ROOT_OVERRIDE in Cell 1.')

ROOT = find_root()
INDEX_PATH = ROOT/'data/processed/c4/manifests/c4_sample_index_1429.csv'
GAUGES_PATH = ROOT/'c4_fusion/private_data/stations_40_PRIVATE.geojson'
OUT_DIR = ROOT/'data/processed/c4/satellite/ascat'
TILE_DIR = OUT_DIR/'cache/daily_native_grid'
for p in [OUT_DIR,TILE_DIR]: p.mkdir(parents=True,exist_ok=True)
for p in [INDEX_PATH,GAUGES_PATH]:
    if not p.exists(): raise FileNotFoundError(f'Missing Notebook 01 source: {p}')

CONFIG = dict(version=SOURCE_VERSION,source='CLMS SWI GLOBAL DAILY V4',collection_id=SWI_COLLECTION_ID,
              max_age_days=MAX_SWI_AGE_DAYS,min_qflag_percent=MIN_QFLAG_PERCENT,
              min_catchment_valid_area=MIN_CATCHMENT_VALID_AREA,
              max_nearest_distance_km=MAX_NEAREST_DISTANCE_KM,
              bands=BANDS,grid='EPSG:4326 0.1 degree edges aligned with global native grid',
              time_policy='product date < forecast date (T-1 or older)')
CONFIG_HASH = hashlib.sha256(json.dumps(CONFIG,sort_keys=True).encode()).hexdigest()

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()

def safe_write(path, payload):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists() and path.read_bytes()==payload:return 'UNCHANGED'
    with tempfile.NamedTemporaryFile(dir=path.parent,prefix='.partial_',delete=False) as t:
        t.write(payload); temporary=Path(t.name)
    os.replace(temporary,path);return 'WRITTEN'

def save_json(path,document):
    return safe_write(path,(json.dumps(document,indent=2,sort_keys=True,allow_nan=False)+'\n').encode())

print('ROOT:', ROOT)
print('Private outputs:', OUT_DIR)
print('RUN_PILOT:', RUN_PILOT, '| RUN_BULK:', RUN_BULK)
print('Collection:',SWI_COLLECTION_ID)

In [2]:
# Cell 2 — 1,429 canonical records; original gauge coordinates and irregular polygon files
index = pd.read_csv(INDEX_PATH,low_memory=False)
required = {'record_id','station_id','forecast_date','split','polygon_repo_path'}
assert required.issubset(index.columns), f'Missing Notebook 01 index columns: {required-set(index.columns)}'
assert len(index)==1429 and index.record_id.is_unique
assert index.station_id.nunique()==39 and index.split.value_counts().to_dict()=={'TRAIN':809,'TEST':351,'VALIDATION':269}
assert (index.record_id == index.station_id.astype(str)+'_'+index.forecast_date.astype(str)).all()
assert index.forecast_date.notna().all()
index['forecast_date']=pd.to_datetime(index['forecast_date'],errors='raise').dt.strftime('%Y-%m-%d')

registry=json.loads(GAUGES_PATH.read_text(encoding='utf8'))
coords={}
for f in registry['features']:
    sid=str(f['properties']['station_id'])
    xy=f['geometry']['coordinates']
    assert f['geometry']['type']=='Point' and len(xy)>=2 and 5<=float(xy[1])<=11 and 79<=float(xy[0])<=83
    assert sid not in coords
    coords[sid]=tuple(map(float,xy[:2]))
assert set(index.station_id).issubset(coords), 'Missing original gauge coordinates in registry.'

polys={}; polygon_paths={}; polygon_hashes={}
for sid, group in index.groupby('station_id'):
    vals=group.polygon_repo_path.dropna().unique()
    assert len(vals)==1, f'{sid}: missing or multiple polygon references'
    relative=str(vals[0]).replace('\\','/')
    assert relative.startswith('shared/spatial/catchments/')
    p=(ROOT/relative).resolve()
    assert p.is_relative_to(ROOT.resolve()) and p.is_file(),f'{sid}: missing polygon {p}'
    d=json.loads(p.read_text(encoding='utf8'))
    feature = d['features'][0] if d.get('type')=='FeatureCollection' else d
    g=shape(feature['geometry'])
    assert g.is_valid and not g.is_empty and g.geom_type in ('Polygon','MultiPolygon'),sid
    assert -180<g.bounds[0]<g.bounds[2]<180 and -90<g.bounds[1]<g.bounds[3]<90
    polys[sid]=g; polygon_paths[sid]=p; polygon_hashes[sid]=sha256(p)
assert len(polys)==39
print('PASS:',len(index),'unique forecast records;',len(polys),'polygon geometries. Only target-free metadata loaded.')
print('Split counts:',index.split.value_counts().to_dict())
print('Pilot:',PILOT_STATION, 'source gauge:',coords[PILOT_STATION])

PASS: 1429 unique forecast records; 39 polygon geometries. Only target-free metadata loaded.
Split counts: {'TRAIN': 809, 'TEST': 351, 'VALIDATION': 269}
Pilot: holombuwa source gauge: (80.2663310003202, 7.18820100028799)


In [ ]:
# Cell 3 — A single EPSG:4326 native-sized (~0.1°) Sri Lanka grid and equal-area overlap weights
# Original SWI V4 global GeoTransform = -180,0.1,0,90,0,-0.1.
# Snap all tile BBOX edges to global tenth-degree cell boundaries.
all_bounds=np.array([g.bounds for g in polys.values()])
xlo=math.floor((all_bounds[:,0].min()-0.30)*10)/10
xhi=math.ceil((all_bounds[:,2].max()+0.30)*10)/10
ylo=math.floor((all_bounds[:,1].min()-0.30)*10)/10
yhi=math.ceil((all_bounds[:,3].max()+0.30)*10)/10
WIDTH=int(round((xhi-xlo)*10));HEIGHT=int(round((yhi-ylo)*10))
BBOX=[round(xlo,1),round(ylo,1),round(xhi,1),round(yhi,1)]
assert 0<WIDTH<=80 and 0<HEIGHT<=100 and WIDTH*HEIGHT<8000,(WIDTH,HEIGHT,BBOX)
TRANSFORM=from_origin(xlo,yhi,0.1,0.1)
assert abs(TRANSFORM.a-0.1)<1e-12 and abs(TRANSFORM.e+0.1)<1e-12
EQ=Transformer.from_crs('EPSG:4326','EPSG:6933',always_xy=True)
GEOD=Geod(ellps='WGS84')

def grid_cell(r,c):
    left=xlo+0.1*c;right=left+0.1
    top=yhi-0.1*r;bottom=top-0.1
    return box(left,bottom,right,top)

# Geometry weights computed once per station; not reprojecting the native raster pixels onto smaller cells.
# For small basins, a partial-cell overlap is the correct support geometry; it is not a new observation.
WEIGHTS={}
for sid,g in polys.items():
    x0,y0,x1,y1=g.bounds
    c0=max(0,int(math.floor((x0-xlo)/0.1))-1);c1=min(WIDTH,int(math.ceil((x1-xlo)/0.1))+1)
    r0=max(0,int(math.floor((yhi-y1)/0.1))-1);r1=min(HEIGHT,int(math.ceil((yhi-y0)/0.1))+1)
    projected=geometry_transform(EQ.transform,g)
    denom=projected.area
    assert denom>0,sid
    cells=[]
    for r in range(r0,r1):
        for c in range(c0,c1):
            chunk=g.intersection(grid_cell(r,c))
            if not chunk.is_empty:
                area=geometry_transform(EQ.transform,chunk).area
                if area>0:cells.append((r,c,area/denom))
    assert cells,f'No native-size cell overlaps {sid}'
    frac=sum(z for _,_,z in cells)
    assert 0.995<=frac<=1.005, f'Polygon/grid intersection fraction {sid} = {frac}'
    WEIGHTS[sid]=cells
print(f'Native-scale Sri Lanka grid {WIDTH} × {HEIGHT} (0.1° each); BBOX (lonlat):',BBOX)
print('Polygon-intersecting grid cells per station:',pd.Series({s:len(v) for s,v in WEIGHTS.items()}).describe().to_dict())

In [4]:
# Cell 4 — Pure local SWI quality assessment, polygon aggregation and explicit nearest-cell fallback

def joint_valid(tile):
    """tile: physical units 8 x HEIGHT x WIDTH, bands ordered by BANDS."""
    assert tile.shape == (8,HEIGHT,WIDTH),(tile.shape,HEIGHT,WIDTH)
    return (np.isfinite(tile[:6]).all(axis=0) &
            (tile[:6]>=0).all(axis=0) & (tile[:6]<=100).all(axis=0) &
            (tile[3:6]>=MIN_QFLAG_PERCENT).all(axis=0) &
            np.isfinite(tile[6]) & (np.abs(tile[6]-1.0)<1e-6) &
            (tile[7]>0.5))

def daily_for_station(sid, date_text, tile):
    good=joint_valid(tile)
    cells=WEIGHTS[sid]
    valid=[(r,c,w) for r,c,w in cells if bool(good[r,c])]
    frac=float(sum(w for _,_,w in valid))
    out=dict(station_id=sid,product_date=date_text,extraction_method='MISSING',
             area_valid_fraction=frac, n_intersecting_grid_cells=len(cells),
             n_valid_intersecting_cells=len(valid),nearest_distance_km=np.nan,
             quality_accepted=False)
    out.update({key:np.nan for key in FEATURES+QUALITY})
    if frac>=MIN_CATCHMENT_VALID_AREA and valid:
        wt=np.array([w for _,_,w in valid],dtype=float)
        vals=np.stack([tile[:6,r,c] for r,c,_ in valid],axis=0)
        means=np.average(vals,axis=0,weights=wt)
        out.update(dict(zip(FEATURES+QUALITY,map(float,means))))
        out['extraction_method']='CATCHMENT_AREA_WEIGHTED'
        out['quality_accepted']=True
    else:
        # Nearest eligible native-size pixel to original WGS84 gauge point, *not* a fabricated catchment mean.
        lon,lat=coords[sid]
        eligible=np.argwhere(good)
        if len(eligible):
            rs=eligible[:,0];cs=eligible[:,1]
            lon2=xlo+(cs+0.5)*0.1
            lat2=yhi-(rs+0.5)*0.1
            _,_,dist=GEOD.inv(np.full(len(cs),lon),np.full(len(cs),lat),lon2,lat2)
            j=int(np.argmin(dist)); km=float(dist[j]/1000)
            if km<=MAX_NEAREST_DISTANCE_KM:
                r,c=int(rs[j]),int(cs[j])
                out.update(dict(zip(FEATURES+QUALITY,map(float,tile[:6,r,c]))))
                out.update(extraction_method='NEAREST_NATIVE_CELL',
                           nearest_distance_km=km,quality_accepted=True)
    if out['quality_accepted']:
        assert all(np.isfinite(out[z]) and 0<=out[z]<=100 for z in FEATURES+QUALITY)
    return out


def tile_paths(product_date):
    p=TILE_DIR/(product_date+'.npz');return p,p.with_suffix('.json')

def cache_tile(product_date):
    path,info_path=tile_paths(product_date)
    if not (path.is_file() and info_path.is_file()):return None
    meta=json.loads(info_path.read_text(encoding='utf8'))
    if meta.get('config_hash')!=CONFIG_HASH or meta.get('date')!=product_date: return None
    if meta.get('npz_sha256')!=sha256(path):raise RuntimeError(f'Cached SWI grid tampering/corruption: {path}')
    with np.load(path,allow_pickle=False) as z: tile=z['bands'].astype('float32')
    assert tile.shape==(8,HEIGHT,WIDTH)
    return tile

def save_tile(product_date, tile, response_hash):
    assert tile.shape==(8,HEIGHT,WIDTH)
    path, info_path=tile_paths(product_date)
    stream=io.BytesIO();np.savez_compressed(stream,bands=tile.astype(np.float32))
    state=safe_write(path,stream.getvalue())
    meta=dict(date=product_date,config_hash=CONFIG_HASH,grid_bbox=BBOX,
              source_response_sha256=response_hash,npz_sha256=sha256(path),
              shape=list(tile.shape),source_collection=SWI_COLLECTION_ID)
    save_json(info_path,meta)
    return state

In [5]:
# Cell 5 — OFFLINE self-tests (no HTTP requests). These must pass before live retrieval.
# Construct a synthetic valid tile and verify weights, missing/fallback logic and time policy.
def product_dates_for(forecast_day):
    origin=date.fromisoformat(str(forecast_day))
    return [(origin-timedelta(days=lag)).isoformat() for lag in range(1,MAX_SWI_AGE_DAYS+1)]

assert product_dates_for('2025-11-17')[0]=='2025-11-16'
assert product_dates_for('2025-11-17')[-1]=='2025-11-10'
assert not any(p>= '2025-11-17' for p in product_dates_for('2025-11-17'))
synthetic=np.zeros((8,HEIGHT,WIDTH),dtype=np.float32)
synthetic[0:3]=np.array([40.,50.,60.])[:,None,None]
synthetic[3:6]=np.array([80.,85.,90.])[:,None,None]
synthetic[6]=1.;synthetic[7]=1.
case=daily_for_station(PILOT_STATION,'2025-11-16',synthetic)
assert case['quality_accepted'] and case['extraction_method']=='CATCHMENT_AREA_WEIGHTED'
assert all(abs(case[k]-v)<1e-5 for k,v in zip(FEATURES,[40,50,60]))
synthetic[3:6]=0
failure=daily_for_station(PILOT_STATION,'2025-11-16',synthetic)
assert not failure['quality_accepted'] and failure['extraction_method']=='MISSING'
print('PASS: polygon native-grid weighting, SWI/QFLAG/SSF quality gating, product-date causality, missing handling.')
print('No remote dataset was accessed.')

PASS: polygon native-grid weighting, SWI/QFLAG/SSF quality gating, product-date causality, missing handling.
No remote dataset was accessed.


In [6]:

# Cell 6 — Copernicus Data Space OAuth and daily Process API tile downloader
# Credentials are loaded locally from secrets.env; no credentials are printed.
# No remote request occurs until access_token() or request_one_day() is called.
#
# Official docs:
# https://documentation.dataspace.copernicus.eu/APIs/SentinelHub/UserGuides/BeginnersGuide.html

# ============================================================
# 6.1 — LOAD CREDENTIALS FROM LOCAL secrets.env
# ============================================================

from pathlib import Path
import os

SECRETS_PATH = ROOT / "c4_fusion" / "notebooks" / "secrets.env"

if not SECRETS_PATH.is_file():
    raise FileNotFoundError(
        f"Cannot find Copernicus credentials file:\n{SECRETS_PATH}\n\n"
        "Create secrets.env with CDSE_CLIENT_ID and CDSE_CLIENT_SECRET."
    )

def load_local_secrets(path):
    """Load two credentials without printing or overwriting existing values."""

    credentials = {}

    for raw_line in Path(path).read_text(encoding="utf-8-sig").splitlines():
        line = raw_line.strip()

        # Ignore empty lines and comments.
        if not line or line.startswith("#"):
            continue

        # Support an optional 'export ' prefix.
        if line.startswith("export "):
            line = line[7:].strip()

        if "=" not in line:
            raise ValueError(
                "Invalid secrets.env entry. Expected KEY=value."
            )

        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip()

        # Remove matching surrounding quotes if present.
        if (
            len(value) >= 2
            and value[0] == value[-1]
            and value[0] in ("'", '"')
        ):
            value = value[1:-1]

        if key in ("CDSE_CLIENT_ID", "CDSE_CLIENT_SECRET"):
            credentials[key] = value

    for key in ("CDSE_CLIENT_ID", "CDSE_CLIENT_SECRET"):
        if not credentials.get(key):
            raise EnvironmentError(
                f"Missing or empty {key} in secrets.env."
            )

        os.environ[key] = credentials[key]

    return True


load_local_secrets(SECRETS_PATH)

print("PASS: Copernicus OAuth credentials loaded from local secrets.env.")
print("Credentials are not displayed.")
print("No network request has been made.")


# ============================================================
# 6.2 — ASCAT SWI V4 EVALSCRIPT
# ============================================================

EVALSCRIPT = r"""
//VERSION=3

function setup() {
    return {
        input: [{
            bands: [
                'SWI001',
                'SWI005',
                'SWI010',
                'QFLAG001',
                'QFLAG005',
                'QFLAG010',
                'SSF',
                'dataMask'
            ]
        }],
        output: {
            id: 'default',
            bands: 8,
            sampleType: 'FLOAT32'
        }
    };
}

function unpack(v, scale) {
    return (v >= 0 && v <= 200) ? v * scale : NaN;
}

function evaluatePixel(s) {
    if (s.dataMask !== 1) {
        return [NaN, NaN, NaN, NaN, NaN, NaN, NaN, 0];
    }

    return [
        unpack(s.SWI001, 0.5),
        unpack(s.SWI005, 0.5),
        unpack(s.SWI010, 0.5),

        unpack(s.QFLAG001, 0.5),
        unpack(s.QFLAG005, 0.5),
        unpack(s.QFLAG010, 0.5),

        s.SSF === 255 ? NaN : s.SSF,
        1
    ];
}
"""


# ============================================================
# 6.3 — OAUTH ACCESS TOKEN
# ============================================================

_TOKEN_CACHE = {
    "value": None,
    "expiry": 0.0
}


def access_token():
    """Retrieve and temporarily cache a Copernicus access token."""

    if (
        _TOKEN_CACHE["value"]
        and time.time() < _TOKEN_CACHE["expiry"] - 90
    ):
        return _TOKEN_CACHE["value"]

    cid = os.environ.get("CDSE_CLIENT_ID", "").strip()
    secret = os.environ.get("CDSE_CLIENT_SECRET", "").strip()

    if not cid or not secret:
        raise EnvironmentError(
            "Missing CDSE_CLIENT_ID or CDSE_CLIENT_SECRET. "
            "Check your local secrets.env file."
        )

    response = requests.post(
        TOKEN_URL,
        data={"grant_type": "client_credentials"},
        auth=(cid, secret),
        timeout=(15, 45)
    )

    response.raise_for_status()

    payload = response.json()

    _TOKEN_CACHE["value"] = payload["access_token"]
    _TOKEN_CACHE["expiry"] = (
        time.time() + int(payload.get("expires_in", 600))
    )

    return _TOKEN_CACHE["value"]


# ============================================================
# 6.4 — DOWNLOAD ONE DAILY ASCAT SWI GRID
# ============================================================

def request_one_day(product_date):
    """Request one daily SWI V4 grid from Copernicus Sentinel Hub."""

    start = date.fromisoformat(product_date)
    end = start + timedelta(days=1)

    task = {
        "input": {
            "bounds": {
                "bbox": BBOX,
                "properties": {
                    "crs": "http://www.opengis.net/def/crs/EPSG/0/4326"
                }
            },
            "data": [
                {
                    "type": "byoc-" + SWI_COLLECTION_ID,
                    "dataFilter": {
                        "timeRange": {
                            "from": start.isoformat() + "T00:00:00Z",
                            "to": end.isoformat() + "T00:00:00Z"
                        }
                    },
                    "processing": {
                        "upsampling": "NEAREST",
                        "downsampling": "NEAREST"
                    }
                }
            ]
        },
        "output": {
            "width": WIDTH,
            "height": HEIGHT,
            "responses": [
                {
                    "identifier": "default",
                    "format": {
                        "type": "image/tiff"
                    }
                }
            ]
        },
        "evalscript": EVALSCRIPT
    }

    response = requests.post(
        PROCESS_URL,
        json=task,
        headers={
            "Authorization": "Bearer " + access_token(),
            "Accept": "image/tiff"
        },
        timeout=(20, 180)
    )

    response.raise_for_status()

    raw = response.content

    if len(raw) > 10_000_000:
        raise ValueError("Unexpected SWI tile response size")

    with MemoryFile(raw) as mem:
        with mem.open() as ds:

            assert ds.count == 8
            assert ds.width == WIDTH
            assert ds.height == HEIGHT

            assert ds.crs.to_epsg() == 4326

            t = ds.transform

            assert abs(t.a - 0.1) < 1e-7
            assert abs(t.e + 0.1) < 1e-7

            assert (
                abs(t.c - xlo) < 1e-6
                and abs(t.f - yhi) < 1e-6
            ), ("Unexpected grid alignment", t)

            tile = ds.read().astype(np.float32)

    # Reject impossible decoded values.
    # Network/API failures must never be recorded as missing SWI.

    bad_values = (
        ((tile[:6] < 0) | (tile[:6] > 100))
        & np.isfinite(tile[:6])
        & (tile[7:8] > 0)
    )

    if bad_values.any():
        raise ValueError(
            "SWI/QFLAG values outside 0..100 after decoding. "
            "Inspect source and scaling."
        )

    return tile, hashlib.sha256(raw).hexdigest()


# ============================================================
# 6.5 — RESTARTABLE DAILY CACHE
# ============================================================

def fetch_day_cached(product_date, allow_remote):
    """Reuse cached data or retrieve and save one missing day."""

    got = cache_tile(product_date)

    if got is not None:
        return got, "CACHE"

    if not allow_remote:
        return None, "NOT_CACHED"

    tile, response_sha256 = request_one_day(product_date)

    save_tile(
        product_date,
        tile,
        response_sha256
    )

    return tile, "REMOTE_SAVED"


print("PASS: ASCAT SWI V4 downloader initialized.")
print("OAuth credentials available; no network request yet.")


PASS: Copernicus OAuth credentials loaded from local secrets.env.
Credentials are not displayed.
No network request has been made.
PASS: ASCAT SWI V4 downloader initialized.
OAuth credentials available; no network request yet.


In [7]:
token = access_token()

print("Copernicus authentication successful:", bool(token))

Copernicus authentication successful: True


In [10]:
# Cell 7 — PILOT: one real daily product + actual Holombuwa catchment QA
pilot_product_day=(date.fromisoformat(PILOT_FORECAST_DATE)-timedelta(days=1)).isoformat()
if not RUN_PILOT:
    print('PILOT OFF. Set RUN_PILOT=True in Cell 1, rerun Cell 1, then rerun Cell 7.')
else:
    tile,status=fetch_day_cached(pilot_product_day,allow_remote=True)
    print('Pilot source:',status,'product date:',pilot_product_day,'tile shape:',tile.shape)
    print('Product decode valid grid-cell fractions:',[round(float(np.isfinite(tile[i]).mean()),3) for i in range(7)])
    print('Surface-state flag counts:',pd.Series(tile[6].ravel()).value_counts(dropna=False).head(8).to_dict())
    print('Joint quality-passing fraction of all national-scale cells:',round(float(joint_valid(tile).mean()),3))
    pilot=daily_for_station(PILOT_STATION,pilot_product_day,tile)
    display(pd.DataFrame([pilot]).T.rename(columns={0:'Value'}))
    if not pilot['quality_accepted']:
        print('REVIEW: Holombuwa failed provisional QA. Do not enable bulk yet; inspect QFLAG/SSF and spatial support.')
    else:
        print('PASS: one real ASCAT SWI001/005/010 pilot with documented extraction method.')
    assert tile[7].shape==(HEIGHT,WIDTH)

Pilot source: REMOTE_SAVED product date: 2025-11-16 tile shape: (8, 34, 24)
Product decode valid grid-cell fractions: [0.609, 0.609, 0.609, 0.609, 0.609, 0.609, 0.609]
Surface-state flag counts: {1.0: 497, nan: 319}
Joint quality-passing fraction of all national-scale cells: 0.609


,Value
station_id,holombuwa
product_date,2025-11-16
extraction_method,CATCHMENT_AREA_WEIGHTED
area_valid_fraction,0.880557
n_intersecting_grid_cells,5
n_valid_intersecting_cells,4
nearest_distance_km,NaN
quality_accepted,True
swi001_percent,39.309434
swi005_percent,46.781248


PASS: one real ASCAT SWI001/005/010 pilot with documented extraction method.


In [30]:
# Cell 8 — Controlled and restartable country-scale daily downloads.
# Repeat Cell 8 only. Each date is downloaded ONCE, then used for all 39 catchments.
first=date.fromisoformat(index.forecast_date.min())-timedelta(days=MAX_SWI_AGE_DAYS)
last=date.fromisoformat(index.forecast_date.max())-timedelta(days=1)
ALL_PRODUCT_DAYS=[(first+timedelta(days=k)).isoformat() for k in range((last-first).days+1)]
missing_days=[d for d in ALL_PRODUCT_DAYS if cache_tile(d) is None]
print('Required distinct product dates:',len(ALL_PRODUCT_DAYS),'from',ALL_PRODUCT_DAYS[0], 'to',ALL_PRODUCT_DAYS[-1])
print('Cached:',len(ALL_PRODUCT_DAYS)-len(missing_days),'| uncached:',len(missing_days))
if not RUN_BULK:
    print('BULK OFF. After a real pilot, set RUN_BULK=True in Cell 1, rerun Cell 1, then Cell 8.')
else:
    for j,pday in enumerate(missing_days[:MAX_NEW_DAYS_PER_RUN],1):
        print(f'[{j}/{min(len(missing_days),MAX_NEW_DAYS_PER_RUN)}] {pday}: starting...',flush=True)
        try:
            t0=time.monotonic()
            tile,status=fetch_day_cached(pday,allow_remote=True)
            print(pday,status,'valid joint cells',int(joint_valid(tile).sum()),
                  'elapsed',round(time.monotonic()-t0,1),'s',flush=True)
        except requests.RequestException as exc:
            print('NETWORK/API ERROR; completed days preserved. Stop and retry later:',type(exc).__name__,str(exc)[:250])
            break
        except KeyboardInterrupt:
            print('Interrupted. Completed day caches remain safe.');raise
    remaining=[d for d in ALL_PRODUCT_DAYS if cache_tile(d) is None]
    print('Now cached:',len(ALL_PRODUCT_DAYS)-len(remaining),'/',len(ALL_PRODUCT_DAYS),
          '| outstanding unique days:',len(remaining))

Required distinct product dates: 45 from 2025-11-10 to 2025-12-24
Cached: 45 | uncached: 0
Now cached: 45 / 45 | outstanding unique days: 0


In [31]:
# Cell 9 — Full daily catchment summaries from the local daily raster caches (NO HTTP)
uncached=[d for d in ALL_PRODUCT_DAYS if cache_tile(d) is None]
if uncached:
    print('NOT READY:',len(uncached),'product days not cached. Sample:',uncached[:8])
    print('Finish Cell 8 first; do NOT manufacture missing products for absent local cache files.')
else:
    daily_rows=[]
    for i,product_day in enumerate(ALL_PRODUCT_DAYS):
        tile=cache_tile(product_day)
        for sid in sorted(polys):
            daily_rows.append(daily_for_station(sid,product_day,tile))
    daily=pd.DataFrame(daily_rows)
    assert len(daily)==len(ALL_PRODUCT_DAYS)*39 and not daily[['station_id','product_date']].duplicated().any()
    assert daily.quality_accepted.eq(daily[FEATURES].notna().all(axis=1)).all()
    print('Daily local extraction ready:',len(daily),'station-product-day rows')
    display(daily.extraction_method.value_counts().to_frame('rows'))
    display(daily[FEATURES+QUALITY].describe().T)

Daily local extraction ready: 1755 station-product-day rows


,rows
extraction_method,
CATCHMENT_AREA_WEIGHTED,1754
NEAREST_NATIVE_CELL,1


,count,mean,std,min,25%,50%,75%,max
swi001_percent,1755.0,63.488111,14.237383,29.291972,53.467793,61.782517,72.996342,99.460200
swi005_percent,1755.0,62.922126,11.125262,36.800282,54.537084,61.704641,71.147769,91.169149
swi010_percent,1755.0,61.768755,9.636757,36.466955,54.341336,60.532646,69.631959,83.636428
qflag001_percent,1755.0,61.892972,12.787215,42.343865,44.500000,70.000000,72.000000,74.000000
qflag005_percent,1755.0,90.032232,4.087546,81.768674,85.000000,92.960829,93.500000,94.000000
qflag010_percent,1755.0,94.737293,2.249524,88.385953,92.000000,96.350606,96.910145,97.000000


In [32]:
# Cell 10 — Align latest quality-accepted T-1 ... T-7 product to each of 1,429 records
if uncached:
    print('BLOCKED: incomplete local daily product coverage. Return to Cell 8.')
else:
    daylookup={(x.station_id,x.product_date):x for x in daily.itertuples(index=False)}
    aligned=[]
    for rec in index.itertuples(index=False):
        selected=None
        for pday in product_dates_for(rec.forecast_date):
            candidate=daylookup[(rec.station_id,pday)]
            if bool(candidate.quality_accepted):
                selected=candidate;break
        base={'record_id':rec.record_id,'station_id':rec.station_id,
              'forecast_date':rec.forecast_date,'split':rec.split,
              'swi_available':selected is not None,
              'swi_source_date':selected.product_date if selected else '',
              'swi_age_days':(date.fromisoformat(rec.forecast_date)-date.fromisoformat(selected.product_date)).days if selected else np.nan,
              'swi_age_logscaled':math.log1p((date.fromisoformat(rec.forecast_date)-date.fromisoformat(selected.product_date)).days)/math.log1p(MAX_SWI_AGE_DAYS) if selected else 0.,
              'swi_validity_flag':1 if selected else 0,
              'extraction_method':selected.extraction_method if selected else 'MISSING',
              'area_valid_fraction':selected.area_valid_fraction if selected else np.nan,
              'nearest_distance_km':selected.nearest_distance_km if selected else np.nan}
        for f in FEATURES+QUALITY:
            base[f]=getattr(selected,f) if selected else np.nan
        aligned.append(base)
    samples=pd.DataFrame(aligned)
    assert len(samples)==1429 and samples.record_id.equals(index.record_id)
    assert (samples[['station_id','forecast_date','split']].values == index[['station_id','forecast_date','split']].values).all()
    assert ((samples.swi_source_date.dropna().replace('',pd.NA).dropna().astype(str)) < samples.loc[samples.swi_available,'forecast_date']).all()
    assert samples.loc[samples.swi_available,'swi_age_days'].between(1,MAX_SWI_AGE_DAYS).all()
    assert samples.loc[~samples.swi_available,'swi_age_days'].isna().all()
    assert (samples.loc[~samples.swi_available,'swi_age_logscaled']==0).all()
    assert samples.loc[samples.swi_available,FEATURES+QUALITY].notna().all().all()
    assert samples.loc[~samples.swi_available,FEATURES+QUALITY].isna().all().all()
    print('PASS: 1429 records aligned without dropping anything or using product dates T or newer.')
    print('Availability:',samples.swi_available.value_counts(dropna=False).to_dict())
    display(samples.groupby('split').agg(n=('record_id','size'),available=('swi_available','sum'),
                                          catchment=('extraction_method',lambda s:(s=='CATCHMENT_AREA_WEIGHTED').sum()),
                                          nearest=('extraction_method',lambda s:(s=='NEAREST_NATIVE_CELL').sum())))

PASS: 1429 records aligned without dropping anything or using product dates T or newer.
Availability: {True: 1429}


,n,available,catchment,nearest
split,,,,
TEST,351,351,351,0
TRAIN,809,809,809,0
VALIDATION,269,269,269,0


In [33]:
# Cell 11 — Final QA, frozen provenance and private exports. No Earth Engine / CDSE calls.
if uncached:
    print('EXPORT BLOCKED: all required daily products must be cached and processed.')
else:
    assert samples.record_id.is_unique and daily[['station_id','product_date']].notna().all().all()
    assert samples.swi_available.eq(samples.swi_validity_flag.eq(1)).all()
    assert len(set(samples.station_id))==39
    assert set(samples.split.unique())=={'TRAIN','VALIDATION','TEST'}
    assert samples.swi_age_days.dropna().between(1,MAX_SWI_AGE_DAYS).all()
    # QA records retained even if no SWI was available; no silent exclusion or target columns.
    rowqa=samples[['record_id','station_id','forecast_date','split','swi_available',
                   'swi_source_date','swi_age_days','extraction_method','area_valid_fraction',
                   'nearest_distance_km']].copy()
    rowqa['id_match']=samples.record_id.eq(index.record_id)
    rowqa['split_match']=samples.split.eq(index.split)
    rowqa['validity_consistent']=samples.swi_available.eq(samples.swi_validity_flag.eq(1))
    rowqa['pre_origin_source']=~samples.swi_available | (samples.swi_source_date < samples.forecast_date)
    rowqa['feature_complete_when_available']=~samples.swi_available | samples[FEATURES+QUALITY].notna().all(axis=1)
    rowqa['missing_unfabricated']=samples.swi_available | samples[FEATURES+QUALITY].isna().all(axis=1)
    passcols=['id_match','split_match','validity_consistent','pre_origin_source',
              'feature_complete_when_available','missing_unfabricated']
    rowqa['qa_pass']=rowqa[passcols].all(axis=1)
    assert rowqa.qa_pass.all(),'Some station-day QA failures—do not export!'
    daily_path=OUT_DIR/'c4_ascat_swi_daily_39_PRIVATE.csv'
    index_path=OUT_DIR/'c4_ascat_swi_sample_index_1429_PRIVATE.csv'
    qa_path=OUT_DIR/'c4_ascat_swi_row_QA_1429_PRIVATE.csv'
    for path,df in [(daily_path,daily),(index_path,samples),(qa_path,rowqa)]:
        status=safe_write(path,df.to_csv(index=False,na_rep='').encode('utf-8'))
        print(status,path.name)
    metadata={'source_version':SOURCE_VERSION,'source_asset':'CLMS SWI V4 daily',
              'collection_id':SWI_COLLECTION_ID,'source_config':CONFIG,'config_hash':CONFIG_HASH,
              'forecast_policy':'Only product dates T-1 through T-7; retrospective availability, NOT live publication proof',
              'spatial_note':'Native-sized 0.1 degree raster sampling and true catchment-area intersections; nearest grid fallback flagged',
              'source_index_sha256':sha256(INDEX_PATH),
              'station_registry_sha256':sha256(GAUGES_PATH),
              'polygon_sha256':polygon_hashes,
              'n_station_days':len(samples),'n_labelled_stations':39,
              'n_required_product_dates':len(ALL_PRODUCT_DAYS),
              'n_available':int(samples.swi_available.sum()),
              'n_missing':int((~samples.swi_available).sum()),
              'feature_order':FEATURES,'qa_feature_order':QUALITY,
              'daily_tile_sha256':{day:sha256(tile_paths(day)[0]) for day in ALL_PRODUCT_DAYS},
              'output_sha256':{p.name:sha256(p) for p in [daily_path,index_path,qa_path]},
              'limitations':['Native-scale satellite model, not local soil probe measurements',
                             'Per-product-day age is nominal calendar lag, not acquisition age',
                             'Full-date historical archive does not prove operational 09:00 availability',
                             '0.1 degree grid loses spatial detail in small catchments',
                             'ASCAT soil moisture less reliable over dense tropical forests, complex terrain or water',
                             'QFLAG >=20%, coverage >=50%, nearest <=25km, max age 7 days are provisional PP1 choices',
                             'Original catchments remain candidate polygons; source coordinate discrepancies not resolved']}
    mf_path=OUT_DIR/'c4_ascat_swi_manifest_PRIVATE.json'
    print(save_json(mf_path,metadata),mf_path.name)
    print('\nFINAL QA: PASS |',int(rowqa.qa_pass.sum()),'/',len(rowqa),'station-day QA rows')
    print('Available:',int(samples.swi_available.sum()),'Missing:',int((~samples.swi_available).sum()))
    display(samples.groupby('split').agg(records=('record_id','size'),available=('swi_available','sum'),
                                         catchment=('extraction_method',lambda s:(s=='CATCHMENT_AREA_WEIGHTED').sum()),
                                         nearest=('extraction_method',lambda s:(s=='NEAREST_NATIVE_CELL').sum()),
                                         missing=('extraction_method',lambda s:(s=='MISSING').sum()),
                                         median_age_days=('swi_age_days','median')))
    display(samples[FEATURES+QUALITY].describe().T)
    print('Per-station missing counts:')
    display(samples.groupby('station_id').agg(records=('record_id','size'),missing=('swi_available',lambda x:int((~x).sum())),
                                              nearest=('extraction_method',lambda x:int((x=='NEAREST_NATIVE_CELL').sum()))).query('missing>0 or nearest>0'))
    print('NOTE: independent field-soil-moisture validation and official catchment QA are outside this notebook.')

WRITTEN c4_ascat_swi_daily_39_PRIVATE.csv
WRITTEN c4_ascat_swi_sample_index_1429_PRIVATE.csv
WRITTEN c4_ascat_swi_row_QA_1429_PRIVATE.csv
WRITTEN c4_ascat_swi_manifest_PRIVATE.json

FINAL QA: PASS | 1429 / 1429 station-day QA rows
Available: 1429 Missing: 0


,records,available,catchment,nearest,missing,median_age_days
split,,,,,,
TEST,351,351,351,0,0,1.0
TRAIN,809,809,809,0,0,1.0
VALIDATION,269,269,269,0,0,1.0


,count,mean,std,min,25%,50%,75%,max
swi001_percent,1429.0,63.243209,13.871619,29.291972,53.602042,61.824076,72.663267,99.460200
swi005_percent,1429.0,63.557738,10.229770,36.800282,55.758213,62.435805,71.144795,91.169149
swi010_percent,1429.0,62.714065,8.864254,40.923338,55.614323,61.090835,70.007265,83.636428
qflag001_percent,1429.0,61.481736,12.884277,42.343865,44.500000,70.000000,72.000000,74.000000
qflag005_percent,1429.0,89.898823,4.115104,81.768674,85.000000,92.774930,93.500000,94.000000
qflag010_percent,1429.0,94.658244,2.265425,88.385953,92.000000,96.234746,96.722689,97.000000


Per-station missing counts:


,records,missing,nearest
station_id,,,


NOTE: independent field-soil-moisture validation and official catchment QA are outside this notebook.
